# Fixing the demand and supply mismatch in the danish energy system

The Danish energy system is facing a significant challenge in balancing supply and demand, especially with the increasing integration of renewable energy sources. This leads to prices that can fluctuate wildly.


## Supply side
More stable supply like nuclear power.

One potential solution to this issue is the implementation of large-scale batteries on a structural level. Like hydro power plants, these batteries can store excess energy generated during periods of low demand and release it during peak demand times. This would help to stabilize the grid and reduce price volatility. This could in the future make the energy prices more stable, but it would require significant investment and infrastructure development. 

(Making the supply curve more flat)


## Demand side
With the current system consumers are incentivized to use energy during off-peak hours when prices are lower, which can help to balance supply and demand. However, this is not a perfect solution, as it relies on consumer behavior and may not be sufficient to address the underlying issues in the energy system.

Another potential solution is for consumers to use energy storage systems, such as home batteries, to store energy when prices are low and use it when prices are high. This can help to reduce demand during peak hours and provide a more stable energy supply. However, this also requires significant investment from consumers and may not be feasible for everyone.

(Making the demand more responsive to price)


# Simulation Environment

We are going to create a simulation environment to model the energy market for a single consumer. 
The consumer is small enough to not affect the market price, as this would otherwise make the problem more complex. 
It takes the market price $p_t$ as given.

We also assume that the consumer knows their own energy consumption patterns in advance and it is $d_t$ for each time period $t$. 
For many consumers, this is a reasonable assumption, as much of energy consumption is predictable (e.g., heating, cooling, appliances). 
Some consumers may have more variable consumption patterns, but this would require a more complex model to capture the uncertainty in demand.

We assume that the demand must be met at all times, meaning that the consumer cannot simply choose to not consume energy during peak hours.
This is a reasonable assumption, as most consumers have essential energy needs that cannot be easily shifted (e.g., heating, cooling, lighting), and if they could be shifted, you could simply shift them to off-peak hours.

The consumer buys $e_t$ kWh of energy from the market at time $t$.

The consumer has a battery of size X kWh.
The battery has a maximum charge and discharge rate of Y kW per hour.
The battery has an efficiency of Z%, meaning that if you charge the battery with 1 kWh, you can only get Z kWh out of it.

The price of the battery should be investigated, as it may not be cost-effective for all consumers to invest in a battery.


Degradation of the battery should also be taken into account, as it may affect the long-term cost-effectiveness of the battery.
Lifetime of the battery should also be considered, as it may affect the long-term cost-effectiveness of the battery.


## Environment


## Agent
Represents a consumer who wants to minimize their energy costs by deciding how much energy to buy from the market and how much to store in the battery.
Takes in state and outputs an action.

## State
Battery level, $b_t$
Market prices, $p_t$
If some demand is unpredictable, we could also include this $d^{\text{unpredictable}}_t$, in the state.
Other factors that could be included in the state to help the agent predict future market prices like weather conditions or economic indicators.

## Action
Each hour decide how much energy to buy, $e_t$.
If bigger than demand, the excess energy is stored in the battery.
If smaller than demand, the deficit is covered by the battery.


## Dynamics
If $e_t > d_t$, then the excess energy is stored in the battery, and the battery level increases by $(e_t - d_t) * Z\%$.
$$b_{t+1} = b_t + (e_t - d_t) * Z\% $$
If $e_t < d_t$, then the deficit is covered by the battery, and the battery level decreases by $d_t - e_t$.
$$b_{t+1} = b_t - (d_t - e_t)$$

The other factors market prices, unpredictable demand, weather conditions, and economic indicators are exogenous and can be modeled as stochastic processes or based on historical data. 

## Feasibility
$0 \leq b_t \leq X$ (battery capacity constraint)
$abs(e_t - d_t) \leq Y$ (charge/discharge rate constraint)



## Reward (cost)
p_t * e_t

## Episode
each day the new markets prices are revealed
each hour new others factors are revealed (e.g., weather conditions, economic indicators)

In [125]:
import numpy as np
import pandas as pd
from pyomo.environ import *

In [126]:
class State:
    def __init__(self, time_step, battery_level, energy_price, carbon_intensity, other_factors=None):
        self.time_step = time_step
        self.battery_level = battery_level
        self.energy_price = energy_price
        self.carbon_intensity = carbon_intensity
        self.other_factors = other_factors


class Action:
    def __init__(self, buy_energy_amount):
        self.buy_energy_amount = buy_energy_amount


class Cost: 
    def __init__(self, amount, carbon_emissions, green_factor):
        self.amount = amount
        self.carbon_emissions = carbon_emissions
        self.weighted_cost = (1 - green_factor) * amount + green_factor * carbon_emissions

class Parameters:
    def __init__(self, initial_battery_level, max_battery_capacity, max_charging_rate, max_discharging_rate, battery_efficiency, green_factor):
        self.initial_battery_level = initial_battery_level
        self.max_battery_capacity = max_battery_capacity
        self.max_charging_rate = max_charging_rate
        self.max_discharging_rate = max_discharging_rate
        self.battery_efficiency = battery_efficiency
        self.green_factor = green_factor


class ExogenousValues:
    def __init__(self, energy_price, carbon_intensity, other_factors=None):
        self.energy_price = energy_price
        self.carbon_intensity = carbon_intensity
        self.other_factors = other_factors


class SimulationResult:
    def __init__(self):
        self.states = []
        self.actions = []
        self.costs = []

    def get_total_cost(self, cost_type):
        return sum(getattr(cost, cost_type) for cost in self.costs)
        

class EvaluationResult:
    def __init__(self, policy_name):
        self.policy_name = policy_name
        self.simulation_results = []

    def print_average_costs(self):
        for cost_type in ['amount', 'carbon_emissions', 'weighted_cost']:
            avg_cost = sum(sim.get_total_cost(cost_type) for sim in self.simulation_results) / len(self.simulation_results)
            print(f"Average {cost_type}: {avg_cost:.2f}")
        




        



class Environment:
    def __init__(self, parameters, demand_profile):
        self.parameters = parameters
        self.demand_profile = demand_profile


    def check_feasibility(self, state, action):
        # return True if action is feasible in the given state, else False
        excess_energy = action.buy_energy_amount - self.demand_profile[state.time_step]
        
        if excess_energy > 0:  # Charging
            if excess_energy > self.parameters.max_charging_rate:
                print("Action exceeds max charging rate.")
                return False
            if state.battery_level + excess_energy * self.parameters.battery_efficiency > self.parameters.max_battery_capacity:
                print("Action exceeds max battery capacity.")
                print(f"Current battery level: {state.battery_level} kWh")
                print(f"Excess energy to charge: {excess_energy} kWh")
                print(f"Charging amount (after efficiency): {excess_energy * self.parameters.battery_efficiency} kWh")
                print(f"Battery level after action: {state.battery_level + excess_energy * self.parameters.battery_efficiency} kWh")
                print(f"Max battery capacity: {self.parameters.max_battery_capacity} kWh")
                return False

        else:  # Discharging
            if abs(excess_energy) > self.parameters.max_discharging_rate:
                print("Action exceeds max discharging rate.")
                print(f"Discharging amount: {abs(excess_energy)} kWh")
                print(f"Max discharging rate: {self.parameters.max_discharging_rate} kWh")
                return False
            if state.battery_level + excess_energy < 0:
                print("Action would deplete battery below zero.")
                print(f"Battery level before action: {state.battery_level} kWh")
                print(f"Excess energy to discharge: {abs(excess_energy)} kWh")
                print(f"Battery level after action: {state.battery_level + excess_energy} kWh")
                return False

        return True

    def get_cost(self, state, action):
        # return cost of taking action in the given state
        energy_cost_amount = action.buy_energy_amount * state.energy_price
        carbon_emissions = action.buy_energy_amount * state.carbon_intensity
        return Cost(amount=energy_cost_amount, carbon_emissions=carbon_emissions, green_factor=self.parameters.green_factor)

    def transition(self, state, action, next_exogen_values):
        # return next state
        excess_energy = action.buy_energy_amount - self.demand_profile[state.time_step]

        # update battery level based on action
        if excess_energy > 0:  # Charging
            new_battery_level = state.battery_level + excess_energy * self.parameters.battery_efficiency
        else:  # Discharging
            new_battery_level = state.battery_level + excess_energy  # excess_energy is negative for discharging

        # get next exogen values
        return State(time_step=state.time_step + 1,
                     battery_level=new_battery_level,
                     energy_price=next_exogen_values.energy_price,
                     carbon_intensity=next_exogen_values.carbon_intensity,
                     other_factors=next_exogen_values.other_factors)
    
    def do_simulation(self, policy, exogen_data):
        result = SimulationResult()

        state = State(time_step=0,
                        battery_level=self.parameters.initial_battery_level,
                        energy_price=exogen_data[0].energy_price,
                        carbon_intensity=exogen_data[0].carbon_intensity,
                        other_factors=exogen_data[0].other_factors)

        for t in range(len(exogen_data)):
            result.states.append(state)

            policy_action = policy.select_action(state)
            result.actions.append(policy_action)
            
            if not self.check_feasibility(state, policy_action):
                raise ValueError(f"Infeasible action selected by policy at time step {t}: {policy_action.buy_energy_amount} kWh")
            
            cost = self.get_cost(state, policy_action)
            result.costs.append(cost)

            if t < len(exogen_data) - 1:  # No need to transition on the last step
                next_exogen_values = exogen_data[t + 1]
                state = self.transition(state, policy_action, next_exogen_values)

        return result
        

    def evaluate_policies(self, policies, all_exogen_data):

        all_results = []
        for policy in policies:
            policy_name = policy.__class__.__name__
            print(f"Evaluating policy: {policy_name}")
            result = EvaluationResult(policy_name)
        
            for sim in range(len(all_exogen_data)):
                exogen_data = all_exogen_data[sim]

                if policy.is_hindsight_policy:
                    policy.see_future_exogen_values(exogen_data)

                simulation_result = self.do_simulation(policy, exogen_data)
                result.simulation_results.append(simulation_result)

            print(f"Policy {policy_name} evaluation completed.")
            result.print_average_costs()
            
            all_results.append(result)

        return all_results


        
        
    

class Agent:
    def __init__(self, environment):
        self.environment = environment
        self.is_hindsight_policy = False  # Set to True for hindsight agents that can see future exogenous values

    def see_future_exogen_values(self, exogen_data):
        raise NotImplementedError("This method should be implemented by hindsight agents that can see future exogenous values")

    def select_action(self, state):
        # return action based on the current state
        raise NotImplementedError("This method should be implemented by subclasses")
    

In [127]:
class DummyAgent(Agent):
    def select_action(self, state):
        return Action(buy_energy_amount=self.environment.demand_profile[state.time_step])
    

In [128]:
class OptimalHindsightAgent(Agent):
    def __init__(self, environment):
        super().__init__(environment)
        self.is_hindsight_policy = True
        self.optimal_actions = None

    def see_future_exogen_values(self, exogen_data):
        model = ConcreteModel()
        # Sets
        time_steps = range(len(exogen_data))
        model.T = Set(initialize=time_steps)

        # Parameters
        model.demand = Param(model.T, initialize={t: self.environment.demand_profile[t] for t in model.T})
        model.energy_price = Param(model.T, initialize={t: exogen_data[t].energy_price for t in model.T})
        model.carbon_intensity = Param(model.T, initialize={t: exogen_data[t].carbon_intensity for t in model.T})

        model.max_battery_capacity = Param(initialize=self.environment.parameters.max_battery_capacity)
        model.max_charging_rate = Param(initialize=self.environment.parameters.max_charging_rate)
        model.max_discharging_rate = Param(initialize=self.environment.parameters.max_discharging_rate)
        model.battery_efficiency = Param(initialize=self.environment.parameters.battery_efficiency)
        model.green_factor = Param(initialize=self.environment.parameters.green_factor)


        # Variables
        model.buy_energy = Var(model.T, within=NonNegativeReals)
        model.charge_battery = Var(model.T, within=NonNegativeReals)
        model.discharge_battery = Var(model.T, within=NonNegativeReals)
        model.battery_level = Var(model.T, within=NonNegativeReals)

        # Objective
        def objective_rule(m):
            return sum(
                (1 - m.green_factor) * m.buy_energy[t] * m.energy_price[t] + 
                m.green_factor * m.buy_energy[t] * m.carbon_intensity[t]
                for t in m.T
            )
        model.objective = Objective(rule=objective_rule, sense=minimize)

        # Constraints
        # Demand must be met at each time step
        def demand_constraint_rule(m, t):
            return m.buy_energy[t] + m.discharge_battery[t] == m.demand[t] + m.charge_battery[t]
        
        model.demand_constraint = Constraint(model.T, rule=demand_constraint_rule)

        # Battery level constraints
        def battery_level_rule(m, t):
            return m.battery_level[t] <= m.max_battery_capacity

        model.battery_capacity_constraint = Constraint(model.T, rule=battery_level_rule)

        # Max charging and discharging rates
        def charging_rate_rule(m, t):
            return m.charge_battery[t] <= m.max_charging_rate
        model.charging_rate_constraint = Constraint(model.T, rule=charging_rate_rule)

        def discharging_rate_rule(m, t):
            return m.discharge_battery[t] <= m.max_discharging_rate
        model.discharging_rate_constraint = Constraint(model.T, rule=discharging_rate_rule)

        def discharging_less_than_battery_rule(m, t):
            return m.discharge_battery[t] <= m.battery_level[t]
        
        model.discharging_battery_constraint = Constraint(model.T, rule=discharging_less_than_battery_rule)

        # Battery level dynamics
        def battery_dynamics_rule(m, t):
            if t == 0:
                return m.battery_level[t] == self.environment.parameters.initial_battery_level
            else:
                return m.battery_level[t] == m.battery_level[t-1] + m.charge_battery[t-1] * m.battery_efficiency - m.discharge_battery[t-1]
        model.battery_dynamics_constraint = Constraint(model.T, rule=battery_dynamics_rule)
        
        # Solve the model
        solver = SolverFactory('gurobi')
        if not solver.available():
            raise RuntimeError("Gurobi solver is not available in your environment.")
        results = solver.solve(model, tee=False)
        status = results.Solver()['Termination condition'].value
        assert status == 'optimal', f'error occurred, status: {status}.  Check model!'

        # save optimal actions for each time step
        self.optimal_actions = {t: Action(buy_energy_amount=round(model.buy_energy[t].value)) for t in model.T}



        

    
    def select_action(self, state):
        return self.optimal_actions[state.time_step]

In [129]:
parameters = Parameters(
    initial_battery_level=0, # Initial battery level in kWh
    max_battery_capacity=100, # Maximum battery capacity in kWh
    max_charging_rate=10, # Maximum charge rate in kW
    max_discharging_rate=10, # Maximum discharge rate in kW
    battery_efficiency=1, # Efficiency of the battery (100%)
    green_factor=0 # Weighting factor for carbon intensity in cost calculation (0 to 1)
)



In [130]:
# demand dict for each hour and day of the week
demand_dict = {}
demand_dict[0] = [20, 18, 15, 10, 5, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95]
demand_dict[1] = [22, 20, 18, 12, 8, 8, 12, 18, 22, 28, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
demand_dict[2] = [25, 22, 20, 15, 10, 10, 15, 22, 25, 30, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 110]
demand_dict[3] = [30, 25, 22, 18, 15, 15, 18, 25, 30, 35, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 110, 120]
demand_dict[4] = [35, 30, 25, 20, 15, 15, 20, 30, 35, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180]
demand_dict[5] = [40, 35, 30, 25, 20, 20, 25, 35, 40, 45, 55, 65, 75, 85, 95, 105, 115, 125, 135, 145, 155, 165, 175, 185]
demand_dict[6] = [30, 25, 20, 15, 10, 10, 15, 25, 30, 35, 45, 55, 65, 75, 85, 95, 105, 115, 125, 135, 145, 155, 165, 175]




In [131]:
df = pd.read_csv("fucking_clean_ass_data_bitch.csv")

all_exogen_data = []
all_demands = []

for zone in ['DK-DK1', 'DK-DK2']:
    zone_data = df[df['zone'] == zone]
    
    zone_exogen_data = []
    zone_demands = []

    for _, row in zone_data.iterrows():
        zone_exogen_data.append(ExogenousValues(
            energy_price=row['value'],
            carbon_intensity=row['carbonIntensity']
        ))
        zone_demands.append(demand_dict[row['day_of_week']][row['hour_of_day']])
    
    all_exogen_data.append(zone_exogen_data)
    all_demands.append(zone_demands)


In [132]:
print(len(all_exogen_data[0]))

17449


In [133]:
environment = Environment(parameters, all_demands[0])

In [134]:
policies = [DummyAgent(environment=environment), OptimalHindsightAgent(environment=environment)]

all_results = environment.evaluate_policies(policies, all_exogen_data)

Evaluating policy: DummyAgent
Policy DummyAgent evaluation completed.
Average amount: 159927088.81
Average carbon_emissions: 166305844.00
Average weighted_cost: 159927088.81
Evaluating policy: OptimalHindsightAgent
Policy OptimalHindsightAgent evaluation completed.
Average amount: 153250421.31
Average carbon_emissions: 164766505.00
Average weighted_cost: 153250421.31
